# Build 03 — Unbiased Performance Evaluation

> *IPS-Corrected Evaluation in the Presence of Selective Labels*

**EN:** Once a loop is detected (Build 02), ordinary metrics are unreliable. This notebook implements IPS (Inverse Propensity Score) weighting — the Horvitz-Thompson estimator — to estimate population-level metrics from selectively labelled data.

**KR:** 루프 탐지 후(Build 02) 일반 메트릭은 신뢰 불가. 이 노트북은 IPS 가중(Horvitz-Thompson 추정량)으로 selectively labelled 데이터에서 모집단 수준 메트릭을 추정.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Propensity Score Estimation

**EN:** Models `P(investigated=1 | features)` via logistic regression. Clipping prevents extreme weights that destabilise the estimator.

**KR:** `P(investigated=1 | features)`를 로지스틱 회귀로 모델링. 클리핑으로 극단 가중치 방지.

In [ ]:
def estimate_propensity_scores(
    df: pd.DataFrame,
    feature_cols: list[str],
    investigation_col: str = 'investigated'
) -> np.ndarray:
    """
    Estimate P(investigated=1 | features) using logistic regression.

    This propensity score is used to:
        (a) Weight investigated claims so they represent the full population
        (b) Correct evaluation metrics for selection bias

    Returns:
        Array of propensity scores for all claims (including uninvestigated)
    """
    X = df[feature_cols].values
    y = df[investigation_col].values.astype(int)

    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)
    propensity_scores = model.predict_proba(X)[:, 1]

    # Clip to avoid extreme weights
    propensity_scores = np.clip(propensity_scores, 0.01, 0.99)

    print(f"Propensity score estimation:")
    print(f"  AUC (how well score predicts investigation): "
          f"{roc_auc_score(y, propensity_scores):.4f}")
    print(f"  Mean propensity: {propensity_scores.mean():.4f}")
    print(f"  Min/Max: {propensity_scores.min():.4f} / {propensity_scores.max():.4f}")

    return propensity_scores

## 3. IPS-Corrected Fraud Rate (Horvitz-Thompson)

In [ ]:
def compute_ips_corrected_fraud_rate(
    df: pd.DataFrame,
    fraud_col: str,
    investigation_col: str,
    propensity_scores: np.ndarray,
) -> float:
    """
    Estimate the true population fraud rate using IPS weighting.

    Standard estimate: P(fraud | investigated)  ← BIASED (higher than true rate)
    IPS estimate:      Σ(fraud_i * w_i) / N     where w_i = 1/propensity_i

    This is the Horvitz-Thompson estimator for missing data.
    """
    investigated_mask = df[investigation_col] == 1
    fraud_labels = df.loc[investigated_mask, fraud_col].values.astype(float)
    weights = 1.0 / propensity_scores[investigated_mask]

    # Horvitz-Thompson estimator
    ips_fraud_rate = np.sum(fraud_labels * weights) / len(df)

    return ips_fraud_rate

## 4. IPS-Corrected AUC (weighted bootstrap)

In [ ]:
def compute_ips_corrected_auc(
    df: pd.DataFrame,
    fraud_col: str,
    score_col: str,
    investigation_col: str,
    propensity_scores: np.ndarray,
) -> float:
    """
    Compute IPS-corrected AUC using weighted ROC.

    Each investigated claim is up-weighted by 1/propensity to simulate
    evaluating on the full (unbiased) population.
    """
    investigated_mask = df[investigation_col] == 1
    sub = df[investigated_mask].copy()
    sub_scores = sub[score_col].values
    sub_labels = sub[fraud_col].values.astype(int)
    sub_weights = 1.0 / propensity_scores[investigated_mask]

    # Weighted AUC (approximate using sklearn with sample_weight)
    try:
        # sklearn's roc_auc_score does not support sample weights directly
        # Use weighted resampling as approximation
        rng = np.random.default_rng(42)
        normalized_weights = sub_weights / sub_weights.sum()
        resample_idx = rng.choice(len(sub), size=len(sub) * 3, replace=True, p=normalized_weights)
        auc = roc_auc_score(sub_labels[resample_idx], sub_scores[resample_idx])
    except Exception:
        auc = np.nan

    return auc

## 5. IPS-Corrected Recall

In [ ]:
def compute_ips_corrected_recall(
    df: pd.DataFrame,
    fraud_col: str,
    score_col: str,
    investigation_col: str,
    propensity_scores: np.ndarray,
    threshold: float = 0.5,
    true_fraud_col: str = None,
) -> dict:
    """
    Estimate true population recall using IPS weighting.

    Standard recall = TP / (TP + FN)
        - Denominator is computed only on investigated claims → biased
        - Assumes no fraud in uninvestigated claims → wrong

    IPS recall:
        - Estimates total fraud in population using IPS
        - Computes recall against that estimated total
    """
    investigated_mask = df[investigation_col] == 1
    sub = df[investigated_mask].copy()
    weights = 1.0 / propensity_scores[investigated_mask]

    fraud_labels = sub[fraud_col].values.astype(int)
    model_preds = (sub[score_col].values >= threshold).astype(int)

    # Standard recall (biased — computed only on investigated)
    standard_recall = recall_score(fraud_labels, model_preds, zero_division=0)

    # IPS-estimated total fraud in population
    ips_total_fraud = np.sum(fraud_labels * weights)
    ips_detected_fraud = np.sum((fraud_labels * model_preds) * weights)
    ips_recall = ips_detected_fraud / ips_total_fraud if ips_total_fraud > 0 else np.nan

    result = {
        'standard_recall': round(standard_recall, 4),
        'ips_corrected_recall': round(ips_recall, 4) if not np.isnan(ips_recall) else None,
        'standard_precision': round(precision_score(fraud_labels, model_preds, zero_division=0), 4),
    }

    if true_fraud_col and true_fraud_col in df.columns:
        true_recall = recall_score(
            df[true_fraud_col].values,
            (df[score_col].values >= threshold).astype(int),
            zero_division=0
        )
        result['true_recall_ground_truth'] = round(true_recall, 4)

    return result

## 6. Full Evaluation Comparison Function

In [ ]:
def full_evaluation_comparison(
    df: pd.DataFrame,
    score_col: str,
    fraud_col: str,
    investigation_col: str,
    feature_cols: list[str],
    true_fraud_col: str = None,
    threshold: float = 0.5,
) -> pd.DataFrame:
    """
    Compare standard biased metrics vs IPS-corrected metrics.

    Returns a summary DataFrame showing the gap between them.
    """
    # Estimate propensity scores
    propensity = estimate_propensity_scores(df, feature_cols, investigation_col)

    # Standard biased evaluation (on investigated claims only)
    investigated_mask = df[investigation_col] == 1
    sub = df[investigated_mask].dropna(subset=[fraud_col])
    labels = sub[fraud_col].astype(int)
    scores = sub[score_col]

    standard_auc = roc_auc_score(labels, scores)
    standard_precision = precision_score(labels, (scores >= threshold).astype(int), zero_division=0)
    standard_recall = recall_score(labels, (scores >= threshold).astype(int), zero_division=0)
    standard_brier = brier_score_loss(labels, scores)
    standard_fraud_rate = labels.mean()

    # IPS-corrected metrics
    ips_fraud_rate = compute_ips_corrected_fraud_rate(df, fraud_col, investigation_col, propensity)
    ips_auc = compute_ips_corrected_auc(df, fraud_col, score_col, investigation_col, propensity)
    ips_recall_results = compute_ips_corrected_recall(
        df, fraud_col, score_col, investigation_col, propensity, threshold, true_fraud_col
    )

    results = {
        'Metric': [
            'Fraud Rate Estimate',
            'AUC-ROC',
            'Precision',
            'Recall',
        ],
        'Standard (biased)': [
            f"{standard_fraud_rate:.4f}",
            f"{standard_auc:.4f}",
            f"{standard_precision:.4f}",
            f"{standard_recall:.4f}",
        ],
        'IPS-Corrected': [
            f"{ips_fraud_rate:.4f}",
            f"{ips_auc:.4f}" if not np.isnan(ips_auc) else "N/A",
            "N/A (unaffected by selection)",
            f"{ips_recall_results['ips_corrected_recall']}",
        ],
    }

    if true_fraud_col and true_fraud_col in df.columns:
        true_labels = df[true_fraud_col]
        true_fraud_rate = true_labels.mean()
        true_auc = roc_auc_score(true_labels, df[score_col])
        true_recall = recall_score(true_labels, (df[score_col] >= threshold).astype(int), zero_division=0)
        results['True (ground truth)'] = [
            f"{true_fraud_rate:.4f}",
            f"{true_auc:.4f}",
            "N/A",
            f"{true_recall:.4f}",
        ]

    comparison_df = pd.DataFrame(results)
    return comparison_df, propensity

## 7. Temporal Holdout Split

In [ ]:
def temporal_holdout_split(
    df: pd.DataFrame,
    date_col: str,
    test_size: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split data by time rather than randomly.

    Why this matters:
        Random split leaks future information into training.
        Temporal split respects the causal order of model deployment.
        If model v1 influenced labels in month 3, we must NOT use month 3
        data to train the model we're evaluating as if it were independent.
    """
    df = df.sort_values(date_col)
    cutoff_idx = int(len(df) * (1 - test_size))
    train = df.iloc[:cutoff_idx].copy()
    test = df.iloc[cutoff_idx:].copy()

    print(f"Temporal split: train={len(train):,} claims, test={len(test):,} claims")
    if date_col in df.columns:
        print(f"  Train period: {df[date_col].iloc[0]} → {df[date_col].iloc[cutoff_idx-1]}")
        print(f"  Test period:  {df[date_col].iloc[cutoff_idx]} → {df[date_col].iloc[-1]}")

    return train, test

---

# 👁️ Section A — Hands-on Demo: Standard vs IPS vs Ground Truth

> *We generate looped data from Build 01, then compute all three flavours of metrics.*

**EN:** The key visual: side-by-side comparison showing how much standard metrics overstate performance, and how much IPS recovers (vs the unattainable ground truth).

**KR:** 핵심 시각: 표준 메트릭이 성능을 얼마나 과대평가하는지, IPS가 얼마나 회복하는지(달성 불가능한 ground truth와 비교).

## A.1 Generate Build-01 looped data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent / '01_sfp_simulation'))
from simulate_sfp_loop import run_full_simulation

print("Generating 20k synthetic claims with 3-generation SFP loop...")
df, _ = run_full_simulation(n_claims=20_000, n_versions=3, epsilon=0.0, seed=42)

# Use v3 as the current model under evaluation
df['investigated'] = df['model_v3_investigated'].astype(int)
df['observed_fraud'] = pd.to_numeric(df['model_v3_observed_fraud'], errors='coerce')

print(f"\n📊 Data ready:")
print(f"  Total claims        : {len(df):,}")
print(f"  Investigation rate  : {df['investigated'].mean():.1%}")
print(f"  Observed fraud rate : {df.loc[df['investigated']==1, 'observed_fraud'].mean():.3f}")
print(f"  True fraud rate     : {df['true_fraud'].mean():.3f}")

## A.2 Step 1 — Estimate propensity scores

**EN:** Fit P(investigated | features). The AUC printed below tells us how strongly the policy is feature-driven — if it's high, IPS weights will be very heterogeneous.

**KR:** P(investigated | features) 적합. 아래 AUC가 정책이 feature 기반인 정도를 보여줌 — 높으면 IPS 가중치가 매우 이질적.

In [ ]:
feature_cols = ['high_amount', 'night_claim', 'high_postcode', 'prior_claims']
propensity = estimate_propensity_scores(df, feature_cols, 'investigated')

print(f"\nPropensity quartiles:")
for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
    print(f"  P{int(q*100):>3}: {np.quantile(propensity, q):.4f}")

## A.3 Step 2 — IPS-corrected fraud rate

**EN:** Compare standard (biased) vs IPS (Horvitz-Thompson) vs ground-truth fraud rate. Standard overstates, IPS recovers.

**KR:** 표준(편향) vs IPS(Horvitz-Thompson) vs ground-truth fraud rate 비교. 표준은 과대, IPS는 회복.

In [ ]:
ips_rate = compute_ips_corrected_fraud_rate(df, 'observed_fraud', 'investigated', propensity)
standard_rate = df.loc[df['investigated']==1, 'observed_fraud'].mean()
true_rate = df['true_fraud'].mean()

print(f"\n📊 Fraud rate estimates:")
print(f"  Standard (investigated only) : {standard_rate:.4f}  ← biased UPWARD")
print(f"  IPS-corrected (Horvitz-Thompson): {ips_rate:.4f}")
print(f"  Ground truth                  : {true_rate:.4f}")
print()
print(f"  Standard error: {(standard_rate - true_rate):+.4f} ({100 * (standard_rate - true_rate) / true_rate:+.1f}%)")
print(f"  IPS error     : {(ips_rate - true_rate):+.4f} ({100 * (ips_rate - true_rate) / true_rate:+.1f}%)")

## A.4 Step 3 — Full evaluation comparison

**EN:** All metrics side-by-side: standard vs IPS vs ground truth. This table is the punch line.

**KR:** 모든 메트릭을 한 표로: 표준 vs IPS vs ground truth. 이 표가 핵심 결론.

In [ ]:
comparison_df, _ = full_evaluation_comparison(
    df=df,
    score_col='model_v3_score',
    fraud_col='observed_fraud',
    investigation_col='investigated',
    feature_cols=feature_cols,
    true_fraud_col='true_fraud',
    threshold=0.5,
)

print("\n📊 EVALUATION COMPARISON")
print("─" * 80)
print(comparison_df.to_string(index=False))

## A.5 Step 4 — Temporal holdout demonstration

In [ ]:
# Build a fake date column for the demo
df['claim_date'] = pd.date_range('2022-01-01', periods=len(df), freq='30min')
train, test = temporal_holdout_split(df, 'claim_date', test_size=0.2)
print(f"\n📊 Split: train fraud rate = {train.loc[train['investigated']==1, 'observed_fraud'].mean():.4f}, "
      f"test fraud rate = {test.loc[test['investigated']==1, 'observed_fraud'].mean():.4f}")

## 🎯 What you should observe

**EN:**
- **Standard fraud rate** > IPS rate > **Ground truth** — standard heavily overstates because we only see fraud where we looked
- **Standard AUC** unrealistically high (often 0.85+) — both train and test are biased by the same policy
- **IPS recall** noticeably lower than standard recall — because the IPS denominator includes estimated unseen fraud
- The **gap between standard and ground truth** is what would be invisible in production

**KR:**
- **표준 fraud rate** > IPS rate > **ground truth** — 표준은 본 곳에서만 fraud를 봤기에 크게 과대평가
- **표준 AUC** 비현실적으로 높음(보통 0.85+) — train과 test 모두 같은 정책으로 편향
- **IPS recall**이 표준 recall보다 낮음 — IPS 분모에 미관측 추정 fraud가 포함되므로
- **표준 vs ground truth 격차**가 프로덕션에서 보이지 않는 부분